# Thermal Solar Fault Segmentation — U-Net Training

---

**Production ML Pipeline for Pixel-Level Thermal Fault Detection**

This notebook trains a U-Net segmentation model to detect faults in thermal images of solar panels.

| Component | Details |
|-----------|--------|
| **Architecture** | U-Net (Encoder-Bottleneck-Decoder with Skip Connections) |
| **Input** | 256x256 RGB Thermal Images |
| **Output** | 256x256 Binary Segmentation Mask |
| **Loss** | Dice Loss + BCE Loss |
| **Optimizer** | AdamW with OneCycleLR |
| **Training** | Mixed Precision (AMP) on GPU |

---

### Dataset Structure (ZIP Upload)

Zip your dataset folder and upload the `.zip` file when prompted.

```
YourDataset.zip
  └── ThermalDataset/
        ├── train/
        │     ├── images/    ← Thermal photos (.jpg/.png)
        │     └── masks/     ← Binary segmentation masks (.png)
        ├── valid/
        │     ├── images/
        │     └── masks/
        └── test/
              ├── images/
              └── masks/
```

---
## 1. Environment Setup
Install dependencies and verify GPU availability.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================

!pip install -q torch torchvision matplotlib opencv-python-headless Pillow numpy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import os
import glob
import time
import random
from pathlib import Path

# ============================================================================
# GPU CHECK
# ============================================================================

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[OK] GPU Detected: {gpu_name}')
    print(f'     VRAM: {gpu_mem:.1f} GB')
    print(f'     CUDA Version: {torch.version.cuda}')
    print(f'     PyTorch: {torch.__version__}')
    torch.backends.cudnn.benchmark = True
    print(f'     cuDNN Benchmark: ON')
else:
    DEVICE = torch.device('cpu')
    print('[WARNING] No GPU detected! Training will be very slow.')
    print('          Go to: Runtime -> Change runtime type -> GPU')

print(f'\nDevice: {DEVICE}')

---
## 2. Upload Dataset (ZIP File)

**Steps:**
1. Zip your `ThermalDataset/` folder into `ThermalDataset.zip`
2. Run the cell below — a file picker will appear
3. Select your ZIP file
4. It will auto-extract and verify the structure

**Expected ZIP contents:**
```
ThermalDataset.zip
  └── ThermalDataset/
        ├── train/
        │     ├── images/
        │     └── masks/
        ├── valid/
        │     ├── images/
        │     └── masks/
        └── test/
              ├── images/
              └── masks/
```

In [ ]:
# ============================================================================
# DATASET UPLOAD — ZIP File Upload + Auto Extract
# ============================================================================

import zipfile
from google.colab import files

print('=' * 60)
print(' UPLOAD YOUR DATASET ZIP FILE')
print('=' * 60)
print('Select your .zip file containing the dataset...\n')

uploaded = files.upload()

# Get the uploaded zip filename
zip_filename = list(uploaded.keys())[0]
print(f'\n[OK] Uploaded: {zip_filename} ({len(uploaded[zip_filename]) / 1024**2:.1f} MB)')

# Extract ZIP
extract_dir = '/content/dataset_extracted'
os.makedirs(extract_dir, exist_ok=True)

print(f'[..] Extracting to {extract_dir} ...')
with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall(extract_dir)
print(f'[OK] Extraction complete!')

# Auto-detect dataset root (find folder containing train/valid or train/test)
DATASET_ROOT = None

for dirpath, dirnames, filenames in os.walk(extract_dir):
    subdirs = set(dirnames)
    # Look for a folder that has train/ and (valid/ or val/)
    if 'train' in subdirs and ('valid' in subdirs or 'val' in subdirs or 'test' in subdirs):
        # Verify train/images exists inside
        if os.path.isdir(os.path.join(dirpath, 'train', 'images')):
            DATASET_ROOT = dirpath
            break

if DATASET_ROOT is None:
    # Fallback: check if extract_dir itself has the structure
    print('\n[!] Could not auto-detect dataset root.')
    print('    Listing extracted contents:\n')
    for item in os.listdir(extract_dir):
        print(f'      {item}/')
    print('\n    Set DATASET_ROOT manually in the next cell if needed.')
    DATASET_ROOT = extract_dir
else:
    print(f'\n[OK] Dataset root detected: {DATASET_ROOT}')

# List what we found
print(f'\nContents of dataset root:')
for item in sorted(os.listdir(DATASET_ROOT)):
    full = os.path.join(DATASET_ROOT, item)
    if os.path.isdir(full):
        sub_items = os.listdir(full)
        print(f'  {item}/ -> {sub_items}')

In [ ]:
# ============================================================================
# VERIFY DATASET STRUCTURE
# ============================================================================

def verify_dataset(root):
    """Verify dataset structure and print statistics."""
    print('=' * 60)
    print(' DATASET VERIFICATION')
    print('=' * 60)

    # Handle val/ vs valid/ naming
    val_folder = 'valid'
    if not os.path.exists(os.path.join(root, 'valid')) and os.path.exists(os.path.join(root, 'val')):
        val_folder = 'val'
        print(f'  [INFO] Using "{val_folder}/" as validation folder\n')

    required = {
        'train/images': 0,
        'train/masks': 0,
        f'{val_folder}/images': 0,
        f'{val_folder}/masks': 0,
        'test/images': 0,
        'test/masks': 0,
    }

    all_ok = True
    for subdir in required:
        full_path = os.path.join(root, subdir)
        if os.path.exists(full_path):
            count = len([f for f in os.listdir(full_path)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            required[subdir] = count
            status = '[OK]' if count > 0 else '[EMPTY]'
            print(f'  {status} {subdir}: {count} files')
            if count == 0:
                all_ok = False
        else:
            print(f'  [MISSING] {subdir}')
            all_ok = False

    print('-' * 60)
    if all_ok:
        train_key = 'train/images'
        val_key = f'{val_folder}/images'
        test_key = 'test/images'
        print('[OK] Dataset structure verified!')
        print(f'     Train: {required[train_key]} images, {required["train/masks"]} masks')
        print(f'     Valid: {required[val_key]} images, {required[f"{val_folder}/masks"]} masks')
        print(f'     Test:  {required[test_key]} images, {required["test/masks"]} masks')
    else:
        print('[ERROR] Dataset structure is incomplete!')
        print('        Make sure your ZIP contains train/, valid/ (or val/), and test/ folders')
        print('        each with images/ and masks/ subfolders.')

    # If val folder is "val", rename to "valid" for consistency downstream
    if val_folder == 'val' and all_ok:
        os.rename(os.path.join(root, 'val'), os.path.join(root, 'valid'))
        print('\n[INFO] Renamed val/ -> valid/ for pipeline consistency')

    return all_ok

dataset_ok = verify_dataset(DATASET_ROOT)

if dataset_ok:
    print(f'\nDATASET_ROOT = "{DATASET_ROOT}"')
    print('Ready to proceed with training!')

---
## 3. Configuration

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

class Config:
    # Data
    IMG_SIZE = 256
    BATCH_SIZE = 16
    NUM_WORKERS = 2

    # Training
    EPOCHS = 40
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-5
    GRADIENT_CLIP = 1.0
    EARLY_STOP_PATIENCE = 10

    # Model
    IN_CHANNELS = 3
    OUT_CHANNELS = 1

    # Paths
    DATASET_ROOT = DATASET_ROOT
    MODEL_SAVE_PATH = '/content/thermal_unet_trained.pth'

cfg = Config()

print('Training Configuration:')
print(f'  Image Size:    {cfg.IMG_SIZE}x{cfg.IMG_SIZE}')
print(f'  Batch Size:    {cfg.BATCH_SIZE}')
print(f'  Epochs:        {cfg.EPOCHS}')
print(f'  Learning Rate: {cfg.LEARNING_RATE}')
print(f'  Weight Decay:  {cfg.WEIGHT_DECAY}')
print(f'  Grad Clip:     {cfg.GRADIENT_CLIP}')
print(f'  Early Stop:    {cfg.EARLY_STOP_PATIENCE} epochs')
print(f'  Device:        {DEVICE}')

---
## 4. Dataset & Data Augmentation

In [ ]:
# ============================================================================
# DATASET CLASS WITH AUGMENTATION
# ============================================================================

class ThermalSegDataset(Dataset):
    """
    Thermal imaging dataset for segmentation.
    Returns (image, mask) pairs resized to IMG_SIZE.
    Applies augmentation on training split only.
    """

    def __init__(self, root, split='train', img_size=256, augment=True):
        self.img_size = img_size
        self.split = split
        self.augment = augment and (split == 'train')

        self.img_dir = os.path.join(root, split, 'images')
        self.mask_dir = os.path.join(root, split, 'masks')

        # Collect all image files
        img_extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
        self.images = []
        for ext in img_extensions:
            self.images.extend(glob.glob(os.path.join(self.img_dir, ext)))
        self.images = sorted(self.images)

        # Filter: keep only images that have a matching mask
        valid_images = []
        for img_path in self.images:
            stem = os.path.splitext(os.path.basename(img_path))[0]
            # Try multiple mask extensions
            for mext in ['.png', '.jpg', '.jpeg', '.bmp']:
                mask_path = os.path.join(self.mask_dir, stem + mext)
                if os.path.exists(mask_path):
                    valid_images.append((img_path, mask_path))
                    break

        self.pairs = valid_images
        print(f'  [{split.upper():>5}] {len(self.pairs)} image-mask pairs loaded')

    def __len__(self):
        return len(self.pairs)

    def apply_augmentation(self, image, mask):
        """Apply synchronized augmentation to image and mask."""

        # Random horizontal flip
        if random.random() > 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        # Random vertical flip
        if random.random() > 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)

        # Random rotation (0, 90, 180, 270)
        if random.random() > 0.5:
            angle = random.choice([90, 180, 270])
            image = TF.rotate(image, angle)
            mask = TF.rotate(mask, angle)

        # Random brightness adjustment (image only)
        if random.random() > 0.5:
            factor = random.uniform(0.8, 1.2)
            image = TF.adjust_brightness(image, factor)

        # Random contrast adjustment (image only)
        if random.random() > 0.5:
            factor = random.uniform(0.8, 1.2)
            image = TF.adjust_contrast(image, factor)

        return image, mask

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        # Load
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        # Resize
        image = TF.resize(image, [self.img_size, self.img_size])
        mask = TF.resize(mask, [self.img_size, self.img_size],
                         interpolation=TF.InterpolationMode.NEAREST)

        # Augmentation (train only)
        if self.augment:
            image, mask = self.apply_augmentation(image, mask)

        # Convert to tensors
        image = TF.to_tensor(image)        # [3, H, W], range [0, 1]
        mask = TF.to_tensor(mask)           # [1, H, W], range [0, 1]
        mask = (mask > 0.5).float()         # Binarize

        return image, mask

In [ ]:
# ============================================================================
# BUILD DATALOADERS
# ============================================================================

print('Loading datasets...')
train_dataset = ThermalSegDataset(cfg.DATASET_ROOT, 'train', cfg.IMG_SIZE, augment=True)
val_dataset   = ThermalSegDataset(cfg.DATASET_ROOT, 'valid', cfg.IMG_SIZE, augment=False)
test_dataset  = ThermalSegDataset(cfg.DATASET_ROOT, 'test',  cfg.IMG_SIZE, augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
    num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
    num_workers=cfg.NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
    num_workers=cfg.NUM_WORKERS, pin_memory=True
)

print(f'\nDataloaders ready:')
print(f'  Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'  Valid: {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'  Test:  {len(test_dataset)} samples, {len(test_loader)} batches')

In [ ]:
# ============================================================================
# VISUALIZE SAMPLE DATA
# ============================================================================

def show_samples(dataset, n=4, title='Dataset Samples'):
    """Display sample image-mask pairs from the dataset."""
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    for i, idx in enumerate(indices):
        image, mask = dataset[idx]

        # Image
        axes[0, i].imshow(image.permute(1, 2, 0).numpy())
        axes[0, i].set_title(f'Image {idx}', fontsize=10)
        axes[0, i].axis('off')

        # Mask
        axes[1, i].imshow(mask.squeeze().numpy(), cmap='hot')
        fault_pct = mask.mean().item() * 100
        axes[1, i].set_title(f'Mask (fault: {fault_pct:.1f}%)', fontsize=10)
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Thermal Image', fontsize=12)
    axes[1, 0].set_ylabel('Ground Truth Mask', fontsize=12)
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, n=4, title='Training Samples (with augmentation)')

---
## 5. U-Net Model Architecture

In [ ]:
# ============================================================================
# U-NET ARCHITECTURE
# ============================================================================

class DoubleConv(nn.Module):
    """(Conv2d -> BatchNorm -> ReLU) x 2 with Dropout."""

    def __init__(self, in_ch, out_ch, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net for binary segmentation.

    Architecture:
        Encoder: 4 downsampling blocks (64 -> 128 -> 256 -> 512)
        Bottleneck: 1024 channels
        Decoder: 4 upsampling blocks with skip connections
        Output: 1-channel logits (use sigmoid for probabilities)

    Input:  [B, 3, 256, 256]
    Output: [B, 1, 256, 256]  (logits)
    """

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Output (logits — no sigmoid here)
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder path
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        # Bottleneck
        b = self.bottleneck(self.pool4(e4))

        # Decoder path with skip connections
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out_conv(d1)

In [ ]:
# ============================================================================
# BUILD MODEL
# ============================================================================

model = UNet(in_channels=cfg.IN_CHANNELS, out_channels=cfg.OUT_CHANNELS).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('U-Net Model Summary:')
print(f'  Total Parameters:     {total_params:>12,}')
print(f'  Trainable Parameters: {trainable_params:>12,}')
print(f'  Model Size:           {total_params * 4 / 1024**2:>10.1f} MB (FP32)')
print(f'  Device:               {DEVICE}')

# Verify with dummy input
with torch.no_grad():
    dummy = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE).to(DEVICE)
    out = model(dummy)
    print(f'\n  Input shape:  {dummy.shape}')
    print(f'  Output shape: {out.shape}')
    print(f'  Output range: [{out.min().item():.3f}, {out.max().item():.3f}] (logits)')

---
## 6. Loss Functions & Metrics

In [ ]:
# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

class DiceLoss(nn.Module):
    """Dice loss for segmentation (operates on logits)."""

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs_flat = probs.contiguous().view(-1)
        targets_flat = targets.contiguous().view(-1)

        intersection = (probs_flat * targets_flat).sum()
        union = probs_flat.sum() + targets_flat.sum()

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice


class CombinedLoss(nn.Module):
    """BCE + Dice Loss (autocast-safe, operates on logits)."""

    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


# ============================================================================
# METRICS
# ============================================================================

def compute_dice_score(logits, targets, threshold=0.5):
    """Compute Dice coefficient from logits."""
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * targets).sum()
    dice = (2.0 * intersection) / (preds.sum() + targets.sum() + 1e-7)
    return dice.item()


def compute_iou_score(logits, targets, threshold=0.5):
    """Compute IoU (Jaccard Index) from logits."""
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    iou = intersection / (union + 1e-7)
    return iou.item()


print('[OK] Loss functions and metrics defined.')
print('     Loss: BCE + Dice (autocast-safe)')
print('     Metrics: Dice Score, IoU Score')

---
## 7. Training Loop

In [ ]:
# ============================================================================
# TRAINING ENGINE
# ============================================================================

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    """Train for one epoch with mixed precision."""
    model.train()
    total_loss = 0.0
    num_samples = 0

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        num_samples += batch_size

    return total_loss / num_samples


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate and compute metrics."""
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    num_samples = 0
    num_batches = 0

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        num_samples += batch_size

        # Per-sample metrics
        for i in range(batch_size):
            total_dice += compute_dice_score(logits[i:i+1], masks[i:i+1])
            total_iou += compute_iou_score(logits[i:i+1], masks[i:i+1])
            num_batches += 1

    avg_loss = total_loss / num_samples
    avg_dice = total_dice / num_batches
    avg_iou = total_iou / num_batches

    return avg_loss, avg_dice, avg_iou


print('[OK] Training engine ready.')

In [ ]:
# ============================================================================
# TRAIN THE MODEL
# ============================================================================

# Loss, optimizer, scheduler, scaler
criterion = CombinedLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=cfg.LEARNING_RATE,
    epochs=cfg.EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=1000.0
)
scaler = torch.amp.GradScaler('cuda')

# History tracking
history = {
    'train_loss': [], 'val_loss': [],
    'val_dice': [], 'val_iou': [], 'lr': []
}
best_val_dice = 0.0
epochs_no_improve = 0

# Print header
print('=' * 100)
print(f' THERMAL U-NET TRAINING — {cfg.EPOCHS} Epochs on {DEVICE}')
print('=' * 100)
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Val Loss":>10} | '
      f'{"Val Dice":>9} | {"Val IoU":>8} | {"Time":>6} | {"LR":>10} | Status')
print('-' * 100)

# Training loop
training_start = time.time()

for epoch in range(1, cfg.EPOCHS + 1):
    epoch_start = time.time()

    # Train
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, DEVICE
    )

    # Validate
    val_loss, val_dice, val_iou = validate(model, val_loader, criterion, DEVICE)

    # Record history
    elapsed = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)
    history['lr'].append(current_lr)

    # Save best model
    status = ''
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': best_val_dice,
            'val_iou': val_iou,
            'val_loss': val_loss,
            'config': vars(cfg),
        }, cfg.MODEL_SAVE_PATH)
        status = '* BEST'
    else:
        epochs_no_improve += 1
        status = f'({epochs_no_improve}/{cfg.EARLY_STOP_PATIENCE})'

    # Print progress
    print(f'{epoch:2d}/{cfg.EPOCHS:2d}  | {train_loss:10.6f} | {val_loss:10.6f} | '
          f'{val_dice:9.4f} | {val_iou:8.4f} | {elapsed:5.1f}s | '
          f'{current_lr:.2e} | {status}')

    # Early stopping
    if epochs_no_improve >= cfg.EARLY_STOP_PATIENCE:
        print(f'\n[!] Early stopping at epoch {epoch} '
              f'(no improvement for {cfg.EARLY_STOP_PATIENCE} epochs)')
        break

# Summary
total_time = time.time() - training_start
print('\n' + '=' * 100)
print(f' TRAINING COMPLETE')
print(f'   Best Val Dice:  {best_val_dice:.4f} ({best_val_dice*100:.2f}%)')
print(f'   Total Time:     {total_time/60:.1f} minutes')
print(f'   Model Saved:    {cfg.MODEL_SAVE_PATH}')
print('=' * 100)

---
## 8. Training Curves

In [ ]:
# ============================================================================
# PLOT TRAINING CURVES
# ============================================================================

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Thermal U-Net Training Curves', fontsize=16, fontweight='bold')

# 1. Loss curves
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss',
             linewidth=2, markersize=4)
axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss',
             linewidth=2, markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss Curves', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# 2. Dice & IoU curves
axes[1].plot(epochs_range, history['val_dice'], 'g-o', label='Val Dice',
             linewidth=2, markersize=4)
axes[1].plot(epochs_range, history['val_iou'], 'b-o', label='Val IoU',
             linewidth=2, markersize=4)
axes[1].axhline(y=best_val_dice, color='g', linestyle='--', alpha=0.5,
                label=f'Best Dice: {best_val_dice:.4f}')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Segmentation Metrics', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

# 3. Learning rate schedule
axes[2].plot(epochs_range, history['lr'], 'm-', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('OneCycleLR Schedule', fontsize=14)
axes[2].grid(True, alpha=0.3)
axes[2].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.savefig('/content/thermal_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('[OK] Training curves saved: /content/thermal_training_curves.png')

---
## 9. Test Inference & Visualization

In [ ]:
# ============================================================================
# LOAD BEST MODEL
# ============================================================================

checkpoint = torch.load(cfg.MODEL_SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('Best model loaded:')
print(f'  Epoch:    {checkpoint["epoch"]}')
print(f'  Val Dice: {checkpoint["val_dice"]:.4f}')
print(f'  Val IoU:  {checkpoint["val_iou"]:.4f}')
print(f'  Val Loss: {checkpoint["val_loss"]:.6f}')

In [ ]:
# ============================================================================
# EVALUATE ON TEST SET
# ============================================================================

test_loss, test_dice, test_iou = validate(model, test_loader, criterion, DEVICE)

print('=' * 50)
print(' TEST SET RESULTS')
print('=' * 50)
print(f'  Test Loss: {test_loss:.6f}')
print(f'  Test Dice: {test_dice:.4f} ({test_dice*100:.2f}%)')
print(f'  Test IoU:  {test_iou:.4f} ({test_iou*100:.2f}%)')
print('=' * 50)

In [ ]:
# ============================================================================
# VISUAL PREDICTIONS ON TEST IMAGES
# ============================================================================

@torch.no_grad()
def visualize_predictions(model, dataset, device, n=6):
    """Show original, ground truth, prediction, and overlay."""
    model.eval()
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    fig, axes = plt.subplots(4, n, figsize=(4 * n, 16))
    fig.suptitle('Test Set Predictions', fontsize=18, fontweight='bold', y=1.01)

    row_labels = ['Input Image', 'Ground Truth', 'Prediction', 'Overlay']

    for col, idx in enumerate(indices):
        image, mask = dataset[idx]
        img_tensor = image.unsqueeze(0).to(device)

        # Predict
        logits = model(img_tensor)
        pred_prob = torch.sigmoid(logits).squeeze().cpu().numpy()
        pred_binary = (pred_prob > 0.5).astype(np.float32)

        img_np = image.permute(1, 2, 0).numpy()
        mask_np = mask.squeeze().numpy()

        # Dice for this sample
        dice = compute_dice_score(
            logits.cpu(), mask.unsqueeze(0)
        )

        # Row 0: Original image
        axes[0, col].imshow(img_np)
        axes[0, col].set_title(f'Sample {idx}', fontsize=10)
        axes[0, col].axis('off')

        # Row 1: Ground truth mask
        axes[1, col].imshow(mask_np, cmap='hot', vmin=0, vmax=1)
        gt_pct = mask_np.mean() * 100
        axes[1, col].set_title(f'GT ({gt_pct:.1f}%)', fontsize=10)
        axes[1, col].axis('off')

        # Row 2: Predicted mask
        axes[2, col].imshow(pred_prob, cmap='hot', vmin=0, vmax=1)
        pred_pct = pred_binary.mean() * 100
        axes[2, col].set_title(f'Pred ({pred_pct:.1f}%) Dice:{dice:.3f}', fontsize=10)
        axes[2, col].axis('off')

        # Row 3: Overlay (image + prediction)
        overlay = img_np.copy()
        fault_mask = pred_binary > 0.5
        overlay[fault_mask] = overlay[fault_mask] * 0.5 + np.array([1, 0, 0]) * 0.5
        axes[3, col].imshow(np.clip(overlay, 0, 1))
        axes[3, col].set_title('Overlay', fontsize=10)
        axes[3, col].axis('off')

    # Row labels
    for i, label in enumerate(row_labels):
        axes[i, 0].set_ylabel(label, fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/content/thermal_test_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('[OK] Predictions saved: /content/thermal_test_predictions.png')


visualize_predictions(model, test_dataset, DEVICE, n=6)

---
## 10. Detailed Test Analysis

In [ ]:
# ============================================================================
# PER-SAMPLE TEST ANALYSIS
# ============================================================================

@torch.no_grad()
def analyze_test_set(model, dataset, device):
    """Compute per-sample Dice and IoU for the full test set."""
    model.eval()
    results = []

    for idx in range(len(dataset)):
        image, mask = dataset[idx]
        img_tensor = image.unsqueeze(0).to(device)
        mask_tensor = mask.unsqueeze(0)

        logits = model(img_tensor)
        dice = compute_dice_score(logits.cpu(), mask_tensor)
        iou = compute_iou_score(logits.cpu(), mask_tensor)
        fault_area = mask.mean().item() * 100

        results.append({'idx': idx, 'dice': dice, 'iou': iou, 'fault_area': fault_area})

    # Summary statistics
    dices = [r['dice'] for r in results]
    ious = [r['iou'] for r in results]

    print('=' * 50)
    print(' TEST SET DETAILED ANALYSIS')
    print('=' * 50)
    print(f'  Samples:    {len(results)}')
    print(f'  Mean Dice:  {np.mean(dices):.4f}')
    print(f'  Std Dice:   {np.std(dices):.4f}')
    print(f'  Min Dice:   {np.min(dices):.4f}')
    print(f'  Max Dice:   {np.max(dices):.4f}')
    print(f'  Mean IoU:   {np.mean(ious):.4f}')
    print('=' * 50)

    # Distribution plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(dices, bins=20, color='green', alpha=0.7, edgecolor='black')
    axes[0].axvline(np.mean(dices), color='red', linestyle='--',
                    label=f'Mean: {np.mean(dices):.4f}')
    axes[0].set_xlabel('Dice Score', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Dice Score Distribution (Test Set)', fontsize=14)
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(ious, bins=20, color='blue', alpha=0.7, edgecolor='black')
    axes[1].axvline(np.mean(ious), color='red', linestyle='--',
                    label=f'Mean: {np.mean(ious):.4f}')
    axes[1].set_xlabel('IoU Score', fontsize=12)
    axes[1].set_ylabel('Count', fontsize=12)
    axes[1].set_title('IoU Distribution (Test Set)', fontsize=14)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return results


test_results = analyze_test_set(model, test_dataset, DEVICE)

---
## 11. Save & Download Model

In [ ]:
# ============================================================================
# SAVE FINAL MODEL
# ============================================================================

# The best model was already saved during training.
# Let's verify and print the final checkpoint details.

final_checkpoint = torch.load(cfg.MODEL_SAVE_PATH, map_location='cpu', weights_only=False)

print('=' * 60)
print(' FINAL MODEL CHECKPOINT')
print('=' * 60)
print(f'  File:      {cfg.MODEL_SAVE_PATH}')
print(f'  Size:      {os.path.getsize(cfg.MODEL_SAVE_PATH) / 1024**2:.1f} MB')
print(f'  Epoch:     {final_checkpoint["epoch"]}')
print(f'  Val Dice:  {final_checkpoint["val_dice"]:.4f} ({final_checkpoint["val_dice"]*100:.2f}%)')
print(f'  Val IoU:   {final_checkpoint["val_iou"]:.4f}')
print(f'  Val Loss:  {final_checkpoint["val_loss"]:.6f}')
print('=' * 60)
print(f'\nCheckpoint contains:')
print(f'  - model_state_dict')
print(f'  - optimizer_state_dict')
print(f'  - training config')
print(f'  - best metrics')

In [ ]:
# ============================================================================
# DOWNLOAD MODEL
# ============================================================================

from google.colab import files

print('Downloading model...')
files.download(cfg.MODEL_SAVE_PATH)
print('[OK] Model downloaded: thermal_unet_trained.pth')

---
## 12. Quick Inference Helper

Use this cell to run inference on any single image.

In [ ]:
# ============================================================================
# SINGLE IMAGE INFERENCE
# ============================================================================

@torch.no_grad()
def predict_single_image(model, image_path, device, img_size=256):
    """
    Run inference on a single thermal image.

    Args:
        model: Trained U-Net model
        image_path: Path to thermal image
        device: torch device
        img_size: resize dimension

    Returns:
        pred_mask: numpy array [H, W] with values 0-1
    """
    model.eval()

    # Load and preprocess
    image = Image.open(image_path).convert('RGB')
    original_size = image.size  # (W, H)
    image_resized = TF.resize(image, [img_size, img_size])
    image_tensor = TF.to_tensor(image_resized).unsqueeze(0).to(device)

    # Predict
    logits = model(image_tensor)
    pred_prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    pred_binary = (pred_prob > 0.5).astype(np.uint8)

    # Fault statistics
    fault_pct = pred_binary.mean() * 100

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(image_resized)
    axes[0].set_title('Input Image', fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(pred_prob, cmap='hot', vmin=0, vmax=1)
    axes[1].set_title(f'Prediction (fault: {fault_pct:.1f}%)', fontsize=14)
    axes[1].axis('off')

    img_np = np.array(image_resized).astype(np.float32) / 255.0
    overlay = img_np.copy()
    fault_mask = pred_binary > 0.5
    overlay[fault_mask] = overlay[fault_mask] * 0.5 + np.array([1, 0, 0]) * 0.5
    axes[2].imshow(np.clip(overlay, 0, 1))
    axes[2].set_title('Overlay', fontsize=14)
    axes[2].axis('off')

    plt.suptitle(f'Thermal Fault Detection — {os.path.basename(image_path)}',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'Fault area: {fault_pct:.1f}%')
    print(f'Mask range: [{pred_prob.min():.3f}, {pred_prob.max():.3f}]')

    return pred_prob


# --- Usage Example ---
# Upload a test image or use one from the test set:
test_images = glob.glob(os.path.join(cfg.DATASET_ROOT, 'test', 'images', '*'))
if test_images:
    sample_path = random.choice(test_images)
    pred = predict_single_image(model, sample_path, DEVICE, cfg.IMG_SIZE)
else:
    print('No test images found. Upload an image and call:')
    print('  predict_single_image(model, "your_image.jpg", DEVICE)')

---

## Training Complete

### Summary

| Component | Details |
|-----------|--------|
| **Model** | U-Net (31M parameters) |
| **Loss** | BCE + Dice (autocast-safe) |
| **Optimizer** | AdamW + OneCycleLR |
| **Training** | Mixed Precision (AMP) |
| **Augmentation** | Flip, Rotate, Brightness, Contrast |
| **Best Checkpoint** | `thermal_unet_trained.pth` |

### Next Steps

1. Download the trained model using the cell above
2. Place it in `backend/models/` of your project
3. Update `thermal_analyzer.py` to load the new weights
4. Run your backend API server

---